# 01 — Pose Extraction on Google Colab (YOLO11-Pose + ByteTrack)

**Project:** CVF25PRJ10 • CVFP26BCPPRJC012  
**Target:** High-Accuracy Pose Extraction with GPU Acceleration (NVIDIA T4 / A100 / V100).  
**Models:** YOLO11m-Pose (`yolo11m-pose.pt`, 640/1280px) + ByteTrack tracker.  
**Đặc điểm an toàn:** Tự động lưu thẳng kết quả vào **Google Drive** và hỗ trợ **Tự động tiếp tục (Auto-Resume)** khi bị ngắt kết nối.


## Colab usage rule

Do not duplicate model/data-processing logic in this notebook. Import reusable code from `src/`. Do not commit raw private videos or large checkpoints to Git.


In [ ]:
# Optional when running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os
import sys

# Tự động clone repository nếu đang chạy trên Google Colab
if Path('/content').is_dir() and not (Path('/content/observable-learning-engagement') / 'src').is_dir():
    print('🚀 Phát hiện Google Colab. Đang tự động clone repository...')
    os.system('git clone -b feature/ver2 https://github.com/hquannnguyen/observable-learning-engagement.git /content/observable-learning-engagement')

if (Path('/content/observable-learning-engagement') / 'src').is_dir():
    os.chdir('/content/observable-learning-engagement')
    if '/content/observable-learning-engagement' not in sys.path:
        sys.path.insert(0, '/content/observable-learning-engagement')

def _is_project_root(path: Path) -> bool:
    return (
        (path / 'AGENTS.md').is_file()
        and (path / 'configs' / 'baseline.yaml').is_file()
        and (path / 'scripts' / 'check_setup.py').is_file()
        and (path / 'src').is_dir()
    )

def find_project_root(start: Path | None = None) -> Path:
    env_root = os.environ.get('CVF_PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if _is_project_root(candidate):
            return candidate
        raise RuntimeError(f'CVF_PROJECT_ROOT is not a valid project root: {candidate}')

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    # Typical Colab case: cwd=/content and the cloned repo is one direct child.
    child_matches = [
        child for child in start.iterdir()
        if child.is_dir() and _is_project_root(child)
    ] if start.is_dir() else []
    if len(child_matches) == 1:
        return child_matches[0]
    if len(child_matches) > 1:
        raise RuntimeError(
            'Multiple project roots found. Set CVF_PROJECT_ROOT explicitly: ' +
            ', '.join(map(str, child_matches))
        )

    # Automatic fallback for Google Colab environment:
    if Path('/content').is_dir() and (start == Path('/content') or Path('/content') in start.parents):
        repo_candidate = Path('/content/observable-learning-engagement')
        if not _is_project_root(repo_candidate):
            import subprocess
            print('🔍 Đang tự động clone repository vào Google Colab (/content)...')
            subprocess.run(
                ['git', 'clone', '-b', 'feature/ver2', 'https://github.com/hquannnguyen/observable-learning-engagement.git', str(repo_candidate)],
                check=False,
            )
        if _is_project_root(repo_candidate):
            return repo_candidate

    raise RuntimeError(
        'Project root not found. Clone/open the repository, or set CVF_PROJECT_ROOT.'
    )

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository root:', ROOT)


## 1. Kiểm tra phần cứng GPU

Trên menu Colab: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** (hoặc A100 GPU).

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
gpu_ok = torch.cuda.is_available()
print('CUDA Available :', gpu_ok)
if gpu_ok:
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    print('⚠️ Đang chạy trên CPU. Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU để tăng tốc!')


## 2. Cài đặt các thư viện cần thiết

Cài đặt Ultralytics, PyArrow, PyYAML và thư viện bám đuôi đối tượng `lap`.

In [ ]:
# Cài đặt đầy đủ các dependencies (kèm lap cho ByteTrack)
!pip install -q ultralytics==8.3.242 pyarrow==20.0.0 pyyaml==6.0.3 lap==0.5.14
print('✅ Đã cài đặt xong toàn bộ thư viện cần thiết!')


## 3. Kết nối Google Drive

Kết nối Google Drive để đọc video và tự động lưu kết quả vĩnh viễn (không bao giờ sợ mất file khi ngắt mạng).

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    print('✅ Google Drive đã kết nối thành công!')
except Exception as e:
    print('Google Drive đã được kết nối trước đó.')


## 4. Tự động xác định đường dẫn Video và Thư mục Lưu kết quả trên Drive

Ô bên dưới sẽ tự động quét toàn bộ Google Drive để tìm thư mục chứa các video `session_001` .. `session_006`.
Nếu bạn nhận thư mục qua link chia sẻ (mục *Được chia sẻ với tôi*), hãy làm theo hướng dẫn in ra trên màn hình nếu chưa thấy video.

In [ ]:
import os
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
search_candidates = [
    drive_root / 'ThiGiacMaytinh' / 'CVF25PRJ10_Data',
    drive_root / 'CVF25PRJ10_Data',
    drive_root / 'ThiGiacMaytinh' / 'Dataset',
    drive_root / 'Dataset',
    drive_root / 'ThiGiacMaytinh',
    drive_root / 'ThiGiacMayTinh',
    drive_root,
    Path('data/raw'),
]

raw_video_dir = None
# 1. Quét nhanh qua các thư mục phổ biến
for cand in search_candidates:
    if cand.is_dir():
        for s in ['session_001', 'session_002']:
            if (cand / s).is_dir() or list(cand.glob(f'**/{s}*')):
                raw_video_dir = cand
                break
    if raw_video_dir:
        break

# 2. Nếu chưa thấy, quét sâu hơn trong MyDrive (tối đa 3 cấp)
if not raw_video_dir and drive_root.is_dir():
    print('🔍 Đang quét sâu Google Drive để tìm video session_001..006...')
    for root, dirs, files in os.walk(drive_root):
        try:
            rel = Path(root).relative_to(drive_root)
            if len(rel.parts) > 3:
                continue
        except Exception:
            continue
        if 'session_001' in dirs or any(f.startswith('session_001') for f in files):
            raw_video_dir = Path(root)
            break

# Thư mục lưu kết quả VĨNH VIỄN thẳng trên Google Drive (không lưu trên ổ tạm của Colab)
if drive_root.is_dir():
    output_poses_dir = drive_root / 'ThiGiacMaytinh' / 'poses'
else:
    output_poses_dir = Path('data/poses')
output_poses_dir.mkdir(parents=True, exist_ok=True)

print('=' * 70)
found_any = False
if raw_video_dir:
    # Kiểm tra xem có video thực sự không
    for s in ['session_001', 'session_002']:
        if (raw_video_dir / s).exists() or list(raw_video_dir.glob(f'**/{s}*')):
            found_any = True
            break

if found_any:
    print('✅ ĐÃ TÌM THẤY VIDEO THÀNH CÔNG!')
    print(f'📁 Thư mục video gốc       : {raw_video_dir}')
    print(f'💾 Thư mục lưu vĩnh viễn Drive: {output_poses_dir}')
else:
    print('⚠️ CHƯA TÌM THẤY THƯ MỤC CHỨA VIDEO SESSION TRÊN GOOGLE DRIVE!')
    print(f'Các thư mục hiện có trong Drive ({drive_root}):')
    if drive_root.is_dir():
        for d in sorted(drive_root.iterdir()):
            if d.is_dir():
                print(f'  📁 {d.name}/')
    print('\n👉 HƯỚNG DẪN 3 BƯỚC KHẮC PHỤC:')
    print('1. Vào https://drive.google.com -> Click mục "Được chia sẻ với tôi" (Shared with me).')
    print('2. Chuột phải vào thư mục dữ liệu (CVF25PRJ10_Data) -> Chọn "Thêm lối tắt vào Drive" (Add shortcut to Drive).')
    print('3. Chọn vị trí lưu là: "Drive của tôi" (My Drive) hoặc bỏ vào trong thư mục "ThiGiacMaytinh".')
    print('Sau khi thêm lối tắt xong, bấm CHẠY LẠI ô này!')
print('=' * 70)


## 5. Chạy trích xuất Pose bằng YOLO11-Pose (Lưu thẳng vào Google Drive + Auto-Resume)

**Tính năng an toàn tuyệt đối:**
- `--output-dir "{output_poses_dir}"`: Cứ xong session nào là file được lưu vĩnh viễn vào Google Drive ngay lập tức!
- `--resume`: Tự động nhận diện session nào đã có trên Drive thì bỏ qua, chỉ chạy tiếp session còn thiếu. Nếu bị ngắt GPU hoặc rớt mạng giữa chừng, lần sau chạy lại sẽ **tiếp tục ngay session dở dang, không mất 1 phút công sức nào**.
- `--imgsz 640`: Tốc độ chạy siêu nhanh (~8 - 10 phút / session 15 phút trên T4 GPU).

In [ ]:
# Đảm bảo đang ở gốc repository và cập nhật mã nguồn mới nhất
%cd /content/observable-learning-engagement
!git pull origin feature/ver2

# Kiểm tra an toàn trước khi chạy
if not raw_video_dir:
    raise RuntimeError("❌ Chưa tìm thấy video! Vui lòng thêm lối tắt (Shortcut) thư mục CVF25PRJ10_Data vào Google Drive theo hướng dẫn ở ô 4 trên.")

# Bấm chạy trích xuất toàn bộ Session 1 đến 6:
!python scripts/batch_extract_sessions.py \
    --raw-dir "{raw_video_dir}" \
    --output-dir "{output_poses_dir}" \
    --resume \
    --imgsz 640 \
    --device cuda


## 6. Thống kê & Kiểm tra các file kết quả đã lưu an toàn trên Drive

In [ ]:
import pandas as pd
from pathlib import Path

pose_dir = Path(output_poses_dir)
files = sorted(pose_dir.glob('session_*_cam_A_poses.parquet'))

print(f'✅ Tìm thấy {len(files)} file Parquet đã lưu an toàn trên Google Drive ({pose_dir}):')
for f in files:
    df = pd.read_parquet(f)
    print(f' • {f.name:<32} | {len(df):>7} detections | {df["track_id"].nunique():>3} unique tracks | {f.stat().st_size / (1024*1024):>5.2f} MB')


## 7. Tùy chọn: Nén và Tải về máy tính

Toàn bộ file đã có sẵn vĩnh viễn trên Google Drive của bạn. Nếu bạn muốn tải thêm 1 bản zip về máy tính cá nhân thì chạy ô bên dưới:

In [ ]:
# Sử dụng zipfile của Python nén trực tiếp từ Drive (không bị lỗi wildcard/FUSE của Linux)
import zipfile
from pathlib import Path

# Tự động tìm thư mục poses chứa các file session
target_dir = Path(output_poses_dir) if 'output_poses_dir' in globals() else Path('/content/drive/MyDrive/ThiGiacMaytinh/poses')
files_to_zip = sorted(list(target_dir.glob('session_*_cam_A_poses.*')))

if not files_to_zip:
    for p_cand in [Path('/content/drive/MyDrive/ThiGiacMaytinh/poses'), Path('/content/drive/MyDrive/ThiGiacMayTinh/poses'), Path('data/poses')]:
        if p_cand.is_dir():
            files_to_zip = sorted(list(p_cand.glob('session_*_cam_A_poses.*')))
            if files_to_zip:
                target_dir = p_cand
                break

zip_filename = 'yolo11_sessions_poses.zip'
if files_to_zip:
    print(f'📦 Đang nén {len(files_to_zip)} file từ: {target_dir}')
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for f in files_to_zip:
            zipf.write(f, arcname=f.name)
            print(f' • Đã nén: {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)')
    print(f'\n✅ Đã tạo file zip thành công: {zip_filename} ({Path(zip_filename).stat().st_size / (1024*1024):.2f} MB)')
    try:
        from google.colab import files
        print('🚀 Đang kích hoạt tải về máy tính qua trình duyệt...')
        files.download(zip_filename)
    except Exception:
        print(f'File nén đã lưu tại: {zip_filename}. Bạn có thể tải thủ công qua panel Files bên trái.')
else:
    print('⚠️ Không tìm thấy file poses nào để nén! Kiểm tra xem thư mục có file .parquet hoặc .csv không.')
